# Module 5.1: Open-Source Email Archiving & `.edu` Mailbox Preservation with Mail-Archiver (`homelab-gpu:5000`)

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 5: Permanent Email Archival & Preservation

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand why graduating students must permanently archive their **`.edu` university email** (job offers, professor recommendation threads, research records, tuition/tax notices) alongside ongoing personal **Gmail / IMAP** backups.
2. Compare leading open-source self-hosted email archiving tools (**Mail-Archiver**, **Open Archiver**, **Bichon**, and headless **`mbsync`**) and select the right tool for your hardware and mailbox providers.
3. Deploy **Mail-Archiver (`s1t5/mailarchiver:latest` + `postgres:17-alpine`)** on `homelab-gpu:5000` with persistent local PostgreSQL storage (automatically dumped to `UNAS 2` nightly in Step 6).
4. Connect both your **Personal Gmail** (via Google App Password over IMAP) and your **University Microsoft 365 / Outlook** mailbox (via Graph API / IMAP, plus a clean `.mbox`/`.pst` fallback if university IT blocks IMAP/OAuth) for searchable, attachment-preserving archival and portable `.eml`/`.mbox` exports.

## 5.1.1 Conceptual Background: Why Self-Host an Email Archive?

Your university `.edu` email account is tied to the exact same expiring Microsoft 365 Education (or Google Workspace for Education) license as your OneDrive. Even for personal Gmail accounts, cloud providers can lock accounts due to false-positive automated abuse flags, compromised passwords, or accidental bulk deletions.

### 1. Comparing Modern Open-Source Email Archivers

| Project | Stack & Footprint | Supported Mailboxes | Why Choose / Skip |
| :--- | :--- | :--- | :--- |
| **Mail-Archiver** (`s1t5/mailarchiver`) | **2 containers** (`mailarchiver` + `postgres:17-alpine`), ~250–400 MB RAM | **Microsoft 365 (Graph API)**, **Personal Outlook (Device Code)**, **Gmail / Generic IMAP**, **MBOX/EML import & export** | **🥇 Best All-Round Homelab Choice (Deployed Here):** Clean searchable web UI, attachment previews, scheduled automated syncs, supports both M365 and Gmail/IMAP natively, and lightweight PostgreSQL backup! |
| **Open Archiver** (`logiclabshq/open-archiver`) | Heavy (Postgres + Redis + Meilisearch + App, ~2–4 GB RAM) | Google Workspace, M365, IMAP, PST/MBOX | Great for enterprise eDiscovery, heavier than needed for personal/family archiving. |
| **Bichon** | Single Rust container (Tantivy index, no SQL DB) | Generic IMAP (Password / OAuth2) | Super fast and lean, though lacks native Microsoft Graph API integration for M365 tenants. |
| **`mbsync` (`isync`) / `imap-backup`** | Headless CLI cron script | Generic IMAP $\rightarrow$ raw Maildir/MBOX files | Zero web search UI—great as a pure raw-file fallback, less pleasant when searching old receipts/threads. |

### 2. Handling University IT Restrictions on `.edu` Mailboxes
Just like University OneDrive, university IT departments sometimes disable legacy IMAP or block third-party Microsoft Graph OAuth apps on `.edu` Outlook accounts:
- **Path A (Direct Sync inside Mail-Archiver):** Connect directly via **Microsoft 365 / IMAP** in Mail-Archiver so every folder (`Inbox`, `Sent`, `Archive`) and every PDF/image attachment syncs automatically on a schedule!
- **Path B (Google Workspace `.edu` Gmail):** Generate a **16-character Google App Password** (`myaccount.google.com/apppasswords` with 2FA enabled) and connect via IMAP (`imap.gmail.com:993`, SSL/TLS)—works cleanly for both personal Gmail and Gmail-backed universities!
- **Path C (Export Fallback When University Blocks All API/IMAP):** Open desktop **Mozilla Thunderbird** (or Outlook desktop) logged into your university mailbox, export your entire `.edu` mailbox to standard **`.mbox`** or **`.eml`** files (or use **Google Takeout** for `.edu` Gmail), store the raw master `.mbox` permanently on `/mnt/unas_media/Email_Archives/`, AND import it into **Mail-Archiver** so it stays searchable in your browser forever!

## 5.1.2 Step-by-Step Laboratory Guide

### Task 5.1.1: Deploy Mail-Archiver (`s1t5/mailarchiver` + `postgres:17-alpine`) on `homelab-gpu:5000`
1. SSH into `homelab-gpu` (`ssh homelab-gpu`) and create the Mail-Archiver stack directory (keeping Postgres data on fast local NVMe—backed up nightly to `UNAS 2` in Module 6.1!):
   ```bash
   mkdir -p ~/homelab/stacks/mail-archiver/{postgres-data,dataprotection-keys}
   cd ~/homelab/stacks/mail-archiver
   ```
2. Generate strong credentials in `.env` (`chmod 600`) and print your generated Web UI admin password so you can save it in your password manager:
   ```bash
   DB_PASS=$(openssl rand -hex 20)
   UI_PASS=$(openssl rand -base64 18 | tr -dc 'A-Za-z0-9' | head -c 20)

   cat << EOT > ~/homelab/stacks/mail-archiver/.env
   MAIL_DB_PASS=${DB_PASS}
   MAIL_UI_USER=admin
   MAIL_UI_PASS=${UI_PASS}
   TZ=America/New_York
   EOT
   chmod 600 ~/homelab/stacks/mail-archiver/.env

   echo "=== Save these Mail-Archiver Web UI Credentials ==="
   echo "URL:      http://10.0.0.201:5000"
   echo "Username: admin"
   echo "Password: ${UI_PASS}"
   ```
3. Create `~/homelab/stacks/mail-archiver/docker-compose.yml`:
   ```bash
   cat << 'EOT' > ~/homelab/stacks/mail-archiver/docker-compose.yml
   name: mail_archiver

   services:
     mailarchiver-postgres:
       image: docker.io/library/postgres:17-alpine
       container_name: mailarchiver_postgres
       restart: unless-stopped
       environment:
         POSTGRES_DB: MailArchiver
         POSTGRES_USER: mailuser
         POSTGRES_PASSWORD: ${MAIL_DB_PASS}
       volumes:
         - ./postgres-data:/var/lib/postgresql/data
       healthcheck:
         test: ["CMD-SHELL", "pg_isready -U mailuser -d MailArchiver"]
         interval: 10s
         timeout: 5s
         retries: 5

     mailarchiver-app:
       image: s1t5/mailarchiver:latest
       container_name: mailarchiver_app
       restart: unless-stopped
       depends_on:
         mailarchiver-postgres:
           condition: service_healthy
       ports:
         - "5000:5000"
       environment:
         - ConnectionStrings__DefaultConnection=Host=mailarchiver-postgres;Database=MailArchiver;Username=mailuser;Password=${MAIL_DB_PASS}
         - Authentication__Username=${MAIL_UI_USER}
         - Authentication__Password=${MAIL_UI_PASS}
         - TimeZone__DisplayTimeZoneId=${TZ}
       volumes:
         - ./dataprotection-keys:/app/DataProtection-Keys
   EOT
   ```
4. Pull and launch the Mail-Archiver stack:
   ```bash
   cd ~/homelab/stacks/mail-archiver
   docker compose pull
   docker compose up -d
   docker compose ps
   ```

### Task 5.1.2: Connect Your Personal Gmail & University `.edu` Mailbox and Run Initial Archive Sync
1. Open your browser and navigate to:
   ```text
   http://10.0.0.201:5000
   ```
   Sign in using `admin` and the password generated in Task 5.1.1.
2. **Add Your Personal Gmail Account (Automated Ongoing Backup):**
   - In your Google Account security settings (`https://myaccount.google.com/apppasswords`), create a dedicated 16-character **App Password** named `Homelab Mail-Archiver`.
   - In Mail-Archiver, click **Mail Accounts $\rightarrow$ Add Account $\rightarrow$ IMAP**:
     - **IMAP Server:** `imap.gmail.com` | **Port:** `993` | **Use SSL/TLS:** Enabled
     - **Username:** `yourname@gmail.com` | **Password:** `<16-char Google App Password>`
     - **Sync Interval:** Every `6` or `12` hours (non-destructive read-only sync—never deletes mail from Gmail!).
3. **Archive Your Graduating University `.edu` Account (Before Account Deactivation!):**
   - **If M365 / IMAP is enabled on your `.edu` tenant:** Add your `.edu` account in Mail-Archiver (`Microsoft 365` or `IMAP`), run a full **Sync Now**, and verify that `Inbox`, `Sent Items`, and key folders finish importing 100% of messages and attachments!
   - **If University IT blocks IMAP/OAuth:** Export your `.edu` mailbox once via **Outlook/Thunderbird (`.mbox` / `.eml`)** or **Google Takeout (`.mbox`)**, save a master copy to `/mnt/unas_media/Email_Archives/University_Edu_Master.mbox`, and import the `.mbox` into Mail-Archiver via **Import** so every old professor email and attachment remains searchable in your browser!
4. Test the full-text search bar in Mail-Archiver (searching for a keyword or PDF attachment name) and verify you can view HTML bodies and download archived attachments cleanly.

### 🖼️ Task 5.1.2 Verification: Mail-Archiver Searchable Inbox & Synced Mailboxes

**Verification Objective:** Capture a screenshot of **Mail-Archiver** (`http://10.0.0.201:5000`) showing your connected account(s) synced with archived message count and working search preview.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] **Mail-Archiver (`s1t5/mailarchiver`) + `postgres:17-alpine`** deployed on `homelab-gpu:5000`.
- [ ] Personal Gmail (via App Password) and/or University `.edu` mailbox synced or imported into Mail-Archiver.
- [ ] Full-text search and attachment previews verified inside the web UI.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 6.1: Local Automated Nightly Backups (`homelab-pi` + `homelab-gpu` $\rightarrow$ `UNAS 2`) & Immich External Library](../06_tiered_backups_google_drive_and_observability/06_1_local_321_backups_and_immich_integration.ipynb).